# Kimi Stage 1 and TriggerSearch: read-only result analysis
Run from the repository root with its native `pipeline.data.load_records` available. This cell only reads the two published scalar projections; it imports no protocol, calls no model, and executes no candidate. No full-study ROC/safety or trigger-catching claim is made. The task-level solve selection is copied from `notebooks/build_results.py` at source revision 411bf4ea2e91ff3a9ff14aaf045ab3c9e39672d1: sort candidate IDs, exclude failed/null solves, and select the first surviving record per task.

In [ ]:
import hashlib, json
from pathlib import Path
from collections import Counter
from pipeline.data import load_records
manifest = json.loads(Path('reports/kimi-stage12-20261024/provenance.json').read_text())
rows = {}
for stage in ('solve', 'trig'):
    name = f'apps1388-411bf4e-20261018-kimi-{stage}'
    path = Path('runs') / name
    assert hashlib.sha256((path/'records.jsonl').read_bytes()).hexdigest() == manifest['runs'][name]['published_records_sha256']
    assert hashlib.sha256((path/'config.json').read_bytes()).hexdigest() == manifest['runs'][name]['config_sha256']
    rows[stage] = load_records(name)
    assert len(rows[stage]) == len({r['candidate_id'] for r in rows[stage]}) == 2776
    assert Counter(r['split'] for r in rows[stage]) == {'train':1110, 'test':1666}
    assert all(not ({'calls','solution','inputs'} & r.keys()) for r in rows[stage])
    print(stage, 'records',len(rows[stage]), 'measured',sum(not r['failed'] for r in rows[stage]), 'failures',dict(Counter(r['blame'] for r in rows[stage] if r['failed'])))
solved_by_task = {}
for record in sorted(rows['solve'], key=lambda r: r['candidate_id']):
    if not record['failed'] and record['solves'] is not None:
        solved_by_task.setdefault(record['task_id'], record)
solved = sum(bool(record['solves']) for record in solved_by_task.values())
print('native task-level solves', solved, '/', len(solved_by_task), '=', solved/len(solved_by_task))
print('candidate-level solves',dict(Counter(r['solves'] for r in rows['solve'] if not r['failed'])))
parsed = [r['n_parsed'] for r in rows['trig'] if not r['failed']]
print('TriggerSearch parsed below/exactly/above requested 30',sum(n<30 for n in parsed),sum(n==30 for n in parsed),sum(n>30 for n in parsed),'range',min(parsed),max(parsed))
assert solved == 1295 and len(solved_by_task) == 1369
assert Counter(r['blame'] for r in rows['solve'] if r['failed']) == {'model':61}
assert Counter(r['blame'] for r in rows['trig'] if r['failed']) == {'model':42}
